# Logistics Data Preprocessing Pipeline
### Week 2: Data Collection, Cleaning, and Preprocessing

**Dataset:** Delivery_Logistics.csv from Kaggle (25,000 deliveries, 15 columns)
**Goal:** Prepare a clean, reliable dataset for logistics analysis.

## Step 1: Data Quality Check

In [2]:
import pandas as pd

df = pd.read_csv("D:\\PROJECT\\Yuva Intern\\Dataset\\Delivery_Logistics.csv")

print("Rows and columns:", df.shape)
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Column types:")
print(df.dtypes)

Rows and columns: (25000, 15)

Missing values per column:
delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km            0
package_weight_kg      0
delivery_time_hours    0
expected_time_hours    0
delayed                0
delivery_status        0
delivery_rating        0
delivery_cost          0
dtype: int64

Duplicate rows: 0

Column types:
delivery_id            float64
delivery_partner        object
package_type            object
vehicle_type            object
delivery_mode           object
region                  object
weather_condition       object
distance_km            float64
package_weight_kg      float64
delivery_time_hours     object
expected_time_hours     object
delayed                 object
delivery_status         object
delivery_rating          int64
delivery_cost          float64
dtype: object


## Data Quality Check Result

| Check | Result |
|---|---|
| Rows and columns | 25,000 rows, 15 columns |
| Missing values | 0 in every column |
| Duplicate rows | 0 |
| Time columns | Stored as text instead of numbers |
| delivery_id | Decimal number with repeated values, not a usable ID |

**Insight:** The dataset has no missing values or duplicate rows, but it has two format problems: the time columns are stored as text, and delivery_id cannot identify a delivery. Both need to be fixed before analysis.

## Step 2: Fix Data Format Problems

In [3]:
# Fix 1: turn the time columns into plain numbers (hours)
df["delivery_time_hours"] = df["delivery_time_hours"].str[-9:].astype(int)
df["expected_time_hours"] = df["expected_time_hours"].str[-9:].astype(int)

# Fix 2: remove the messy delivery_id column
df = df.drop(columns=["delivery_id"])

print("Rows and columns:", df.shape)
print()
print(df.dtypes)

Rows and columns: (25000, 14)

delivery_partner        object
package_type            object
vehicle_type            object
delivery_mode           object
region                  object
weather_condition       object
distance_km            float64
package_weight_kg      float64
delivery_time_hours      int64
expected_time_hours      int64
delayed                 object
delivery_status         object
delivery_rating          int64
delivery_cost          float64
dtype: object


## Format Fix Result

| Fix | Before | After |
|---|---|---|
| Time columns | Text (object) | Whole numbers (int64) |
| delivery_id | Present, with repeated values | Removed |
| Columns in dataset | 15 | 14 |

**Insight:** The time columns can now be used for calculations such as averages and delay checks. Removing delivery_id avoids using a column that could mislead the analysis.

## Step 3: Handle Missing Values (Simulation)

In [4]:
import numpy as np

# Make a copy so the real data stays untouched
df_test = df.copy()

# Remove 5% of the distance values on purpose (1,250 rows)
np.random.seed(42)
rows = np.random.choice(df_test.index, size=1250, replace=False)
df_test.loc[rows, "distance_km"] = np.nan

print("Missing before fix:", df_test["distance_km"].isnull().sum())

# Fix: fill the gaps with the median distance
median_distance = df_test["distance_km"].median()
df_test["distance_km"] = df_test["distance_km"].fillna(median_distance)

print("Missing after fix:", df_test["distance_km"].isnull().sum())
print("Median used:", round(median_distance, 1))

Missing before fix: 1250
Missing after fix: 0
Median used: 151.2


## Missing Values Result

| Check | Result |
|---|---|
| Missing distance values created (simulated) | 1,250 (5% of the data) |
| Missing values after fix | 0 |
| Method used | Fill with the median |
| Median distance used | 151.2 km |

**Insight:** The original dataset had no missing values, so a copy was used to demonstrate the technique. The median was chosen because it is not affected by very large or very small values. Filling the gaps keeps all 25,000 rows available for analysis.

## Step 4: Detect Outliers

In [5]:
columns = ["distance_km", "package_weight_kg", "delivery_cost"]

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(col)
    print("  Normal range:", round(lower, 1), "to", round(upper, 1))
    print("  Outliers found:", len(outliers))
    print()

distance_km
  Normal range: -147.6 to 448.4
  Outliers found: 0

package_weight_kg
  Normal range: -24.8 to 75.1
  Outliers found: 0

delivery_cost
  Normal range: -629.9 to 2358.6
  Outliers found: 0



## Outlier Detection Result

| Column | Normal range (IQR method) | Outliers found |
|---|---|---|
| distance_km | -147.6 to 448.4 | 0 |
| package_weight_kg | -24.8 to 75.1 | 0 |
| delivery_cost | -629.9 to 2,358.6 | 0 |

**Insight:** No values fall outside the normal range, so all 25,000 rows were kept. If outliers had been found, they could be checked first and then either removed or capped at the upper and lower limits, so a few extreme values do not distort averages and models.

## Step 5: Normalize the Data

In [6]:
from sklearn.preprocessing import MinMaxScaler

columns = ["distance_km", "package_weight_kg", "delivery_cost"]

# Make a copy so the original values stay safe
df_scaled = df.copy()

# Scale the three columns to the range 0 to 1
scaler = MinMaxScaler()
df_scaled[columns] = scaler.fit_transform(df[columns])

print("Before:")
print(df[columns].describe().loc[["min", "max"]].round(1))
print()
print("After:")
print(df_scaled[columns].describe().loc[["min", "max"]].round(1))

Before:
     distance_km  package_weight_kg  delivery_cost
min          3.6                0.7           95.7
max        297.1               49.5         1632.7

After:
     distance_km  package_weight_kg  delivery_cost
min          0.0                0.0            0.0
max          1.0                1.0            1.0


## Normalization Result

| Column | Before (min to max) | After (min to max) |
|---|---|---|
| distance_km | 3.6 to 297.1 | 0.0 to 1.0 |
| package_weight_kg | 0.7 to 49.5 | 0.0 to 1.0 |
| delivery_cost | 95.7 to 1,632.7 | 0.0 to 1.0 |

**Insight:** The three columns had very different scales. After Min-Max scaling, all of them lie between 0 and 1, so no column is treated as more important just because its numbers are larger. A copy was used, so the original values are still available for reporting.

## Step 6: Check Text Consistency and Save the Clean Data

In [7]:
text_columns = ["delivery_partner", "package_type", "vehicle_type",
                "delivery_mode", "region", "weather_condition",
                "delayed", "delivery_status"]

for col in text_columns:
    print(col, ":", sorted(df[col].unique()))

# Save the cleaned data to a new file
df.to_csv("Delivery_Logistics_Cleaned.csv", index=False)
print()
print("Clean file saved. Rows and columns:", df.shape)

delivery_partner : ['amazon logistics', 'blue dart', 'delhivery', 'dhl', 'ecom express', 'ekart', 'fedex', 'shadowfax', 'xpressbees']
package_type : ['automobile parts', 'clothing', 'cosmetics', 'documents', 'electronics', 'fragile items', 'furniture', 'groceries', 'pharmacy']
vehicle_type : ['bike', 'ev bike', 'ev van', 'scooter', 'truck', 'van']
delivery_mode : ['express', 'same day', 'standard', 'two day']
region : ['central', 'east', 'north', 'south', 'west']
weather_condition : ['clear', 'cold', 'foggy', 'hot', 'rainy', 'stormy']
delayed : ['no', 'yes']
delivery_status : ['delayed', 'delivered', 'failed']

Clean file saved. Rows and columns: (25000, 14)


## Text Consistency Result

| Column | Different values found |
|---|---|
| delivery_partner | 9 (for example delhivery, ekart, dhl, fedex) |
| package_type | 9 (for example clothing, electronics, groceries) |
| vehicle_type | 6 (bike, ev bike, ev van, scooter, truck, van) |
| delivery_mode | 4 (express, same day, standard, two day) |
| region | 5 (central, east, north, south, west) |
| weather_condition | 6 (clear, cold, foggy, hot, rainy, stormy) |
| delayed | 2 (yes, no) |
| delivery_status | 3 (delivered, delayed, failed) |

**Insight:** All text values are already in lowercase with no spelling differences, so no changes were needed. If "Express" and "express" had both appeared, they would have been counted as two different groups and given wrong results. The cleaned dataset was saved as Delivery_Logistics_Cleaned.csv with 25,000 rows and 14 columns.